<a href="https://colab.research.google.com/github/ReverseScore/notebooks/blob/main/tango_mt3_transcription.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tango Transcription with Transformers (MT3 + Demucs)

This notebook transcribes Argentine tango recordings into MIDI using:

1. **demucs** (`htdemucs_6s`) to separate the audio into stems.
2. **Google MT3** transformer models to transcribe each stem:
   - `ismir2021` for the **piano** stem (piano-only, with velocities).
   - `mt3` for the **bandoneón/violin**, **bass**, and **drums** stems.

**Caveats**:
- MT3 is **not trained on singing**, so the voice stem is skipped by default.
- Bandoneón is not a standard General-MIDI instrument; it is mapped to **accordion (program 21)**.
- The `other` stem from demucs contains both bandoneón and violin together; MT3 will label them heuristically.

### Instructions
1. Set a GPU runtime: `Runtime` → `Change runtime type` → `GPU`.
2. Run each cell in order.
3. Upload your audio when prompted.
4. Download the combined MIDI (and optionally per-stem MIDIs).

In [ ]:
#@title Setup Environment
#@markdown Install demucs, MT3, and dependencies (may take a few minutes).

!apt-get update -qq && apt-get install -qq libfluidsynth3 build-essential libasound2-dev libjack-dev ffmpeg

# Install demucs for source separation.
!python3 -m pip install -q demucs

# Install MT3 from GitHub.
!git clone --branch=main --depth=1 https://github.com/magenta/mt3 /content/mt3
!python3 -m pip install -q jax[cuda12] nest-asyncio pyfluidsynth==1.3.0 -e /content/mt3 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

# Download MT3 checkpoints and soundfont.
!gsutil -q -m cp -r gs://mt3/checkpoints /content/checkpoints
!gsutil -q -m cp gs://magentadata/soundfonts/SGM-v2.01-Sal-Guit-Bass-V1.3.sf2 /content/SGM-v2.01-Sal-Guit-Bass-V1.3.sf2

print("Setup complete.")

In [ ]:
#@title Imports and Definitions

import functools
import os
import glob
from pathlib import Path

import numpy as np
import tensorflow.compat.v2 as tf
import gin
import jax
import librosa
import note_seq
import seqio
import t5
import t5x

from mt3 import metrics_utils
from mt3 import models
from mt3 import network
from mt3 import note_sequences
from mt3 import preprocessors
from mt3 import spectrograms
from mt3 import vocabularies

from google.colab import files

import nest_asyncio
nest_asyncio.apply()

SAMPLE_RATE = 16000
SF2_PATH = 'SGM-v2.01-Sal-Guit-Bass-V1.3.sf2'

# General MIDI programs for tango instruments.
TANGO_PROGRAMS = {
    'piano': 0,       # Acoustic Grand Piano
    'bandoneon': 21,  # Accordion is the closest GM match
    'violin': 40,     # Violin
    'bass': 32,       # Acoustic Bass
    'drums': 0,       # Program is ignored when is_drum=True
    'voice': 91,      # Choir Aahs (MT3 is not trained on voice)
}

def upload_audio(sample_rate=SAMPLE_RATE):
    """Upload audio file and return samples at target sample rate."""
    data = list(files.upload().values())
    if len(data) > 1:
        print('Multiple files uploaded; using the first one.')
    return note_seq.audio_io.wav_data_to_samples_librosa(data[0], sample_rate=sample_rate)


class InferenceModel(object):
    """Wrapper of T5X model for music transcription."""

    def __init__(self, checkpoint_path, model_type='mt3'):
        if model_type == 'ismir2021':
            num_velocity_bins = 127
            self.encoding_spec = note_sequences.NoteEncodingSpec
            self.inputs_length = 512
        elif model_type == 'mt3':
            num_velocity_bins = 1
            self.encoding_spec = note_sequences.NoteEncodingWithTiesSpec
            self.inputs_length = 256
        else:
            raise ValueError('unknown model_type: %s' % model_type)

        gin_files = ['/content/mt3/gin/model.gin',
                     f'/content/mt3/gin/{model_type}.gin']

        self.batch_size = 8
        self.outputs_length = 1024
        self.sequence_length = {'inputs': self.inputs_length,
                                'targets': self.outputs_length}

        self.partitioner = t5x.partitioning.PjitPartitioner(num_partitions=1)

        self.spectrogram_config = spectrograms.SpectrogramConfig()
        self.codec = vocabularies.build_codec(
            vocab_config=vocabularies.VocabularyConfig(
                num_velocity_bins=num_velocity_bins))
        self.vocabulary = vocabularies.vocabulary_from_codec(self.codec)
        self.output_features = {
            'inputs': seqio.ContinuousFeature(dtype=tf.float32, rank=2),
            'targets': seqio.Feature(vocabulary=self.vocabulary),
        }

        self._parse_gin(gin_files)
        self.model = self._load_model()
        self.restore_from_checkpoint(checkpoint_path)

    @property
    def input_shapes(self):
        return {
            'encoder_input_tokens': (self.batch_size, self.inputs_length),
            'decoder_input_tokens': (self.batch_size, self.outputs_length)
        }

    def _parse_gin(self, gin_files):
        gin_bindings = [
            'from __gin__ import dynamic_registration',
            'from mt3 import vocabularies',
            'VOCAB_CONFIG=@vocabularies.VocabularyConfig()',
            'vocabularies.VocabularyConfig.num_velocity_bins=%NUM_VELOCITY_BINS'
        ]
        with gin.unlock_config():
            gin.parse_config_files_and_bindings(
                gin_files, gin_bindings, finalize_config=False)

    def _load_model(self):
        model_config = gin.get_configurable(network.T5Config)()
        module = network.Transformer(config=model_config)
        return models.ContinuousInputsEncoderDecoderModel(
            module=module,
            input_vocabulary=self.output_features['inputs'].vocabulary,
            output_vocabulary=self.output_features['targets'].vocabulary,
            optimizer_def=t5x.adafactor.Adafactor(decay_rate=0.8, step_offset=0),
            input_depth=spectrograms.input_depth(self.spectrogram_config))

    def restore_from_checkpoint(self, checkpoint_path):
        train_state_initializer = t5x.utils.TrainStateInitializer(
            optimizer_def=self.model.optimizer_def,
            init_fn=self.model.get_initial_variables,
            input_shapes=self.input_shapes,
            partitioner=self.partitioner)

        restore_checkpoint_cfg = t5x.utils.RestoreCheckpointConfig(
            path=checkpoint_path, mode='specific', dtype='float32')

        train_state_axes = train_state_initializer.train_state_axes
        self._predict_fn = self._get_predict_fn(train_state_axes)
        self._train_state = train_state_initializer.from_checkpoint_or_scratch(
            [restore_checkpoint_cfg], init_rng=jax.random.PRNGKey(0))

    @functools.lru_cache()
    def _get_predict_fn(self, train_state_axes):
        def partial_predict_fn(params, batch, decode_rng):
            return self.model.predict_batch_with_aux(
                params, batch, decoder_params={'decode_rng': None})
        return self.partitioner.partition(
            partial_predict_fn,
            in_axis_resources=(
                train_state_axes.params,
                t5x.partitioning.PartitionSpec('data',), None),
            out_axis_resources=t5x.partitioning.PartitionSpec('data',)
        )

    def predict_tokens(self, batch, seed=0):
        prediction, _ = self._predict_fn(
            self._train_state.params, batch, jax.random.PRNGKey(seed))
        return self.vocabulary.decode_tf(prediction).numpy()

    def __call__(self, audio):
        ds = self.audio_to_dataset(audio)
        ds = self.preprocess(ds)
        model_ds = self.model.FEATURE_CONVERTER_CLS(pack=False)(
            ds, task_feature_lengths=self.sequence_length)
        model_ds = model_ds.batch(self.batch_size)

        inferences = (tokens for batch in model_ds.as_numpy_iterator()
                      for tokens in self.predict_tokens(batch))

        predictions = []
        for example, tokens in zip(ds.as_numpy_iterator(), inferences):
            predictions.append(self.postprocess(tokens, example))

        result = metrics_utils.event_predictions_to_ns(
            predictions, codec=self.codec, encoding_spec=self.encoding_spec)
        return result['est_ns']

    def audio_to_dataset(self, audio):
        frames, frame_times = self._audio_to_frames(audio)
        return tf.data.Dataset.from_tensors({
            'inputs': frames,
            'input_times': frame_times,
        })

    def _audio_to_frames(self, audio):
        frame_size = self.spectrogram_config.hop_width
        padding = [0, frame_size - len(audio) % frame_size]
        audio = np.pad(audio, padding, mode='constant')
        frames = spectrograms.split_audio(audio, self.spectrogram_config)
        num_frames = len(audio) // frame_size
        times = np.arange(num_frames) / self.spectrogram_config.frames_per_second
        return frames, times

    def preprocess(self, ds):
        pp_chain = [
            functools.partial(
                t5.data.preprocessors.split_tokens_to_inputs_length,
                sequence_length=self.sequence_length,
                output_features=self.output_features,
                feature_key='inputs',
                additional_feature_keys=['input_times']),
            preprocessors.add_dummy_targets,
            functools.partial(
                preprocessors.compute_spectrograms,
                spectrogram_config=self.spectrogram_config)
        ]
        for pp in pp_chain:
            ds = pp(ds)
        return ds

    def postprocess(self, tokens, example):
        tokens = self._trim_eos(tokens)
        start_time = example['input_times'][0]
        start_time -= start_time % (1 / self.codec.steps_per_second)
        return {
            'est_tokens': tokens,
            'start_time': start_time,
            'raw_inputs': []
        }

    @staticmethod
    def _trim_eos(tokens):
        tokens = np.array(tokens, np.int32)
        if vocabularies.DECODED_EOS_ID in tokens:
            tokens = tokens[:np.argmax(tokens == vocabularies.DECODED_EOS_ID)]
        return tokens


def load_audio(path, sample_rate=SAMPLE_RATE):
    """Load a WAV/MP3 file to a 1-D float32 array."""
    return note_seq.audio_io.wav_data_to_samples_librosa(
        open(path, 'rb').read(), sample_rate=sample_rate)


def assign_program(ns, program, is_drum=False):
    """Set the same MIDI program for all notes in a NoteSequence."""
    for note in ns.notes:
        note.program = program
        note.is_drum = is_drum
    return ns


def merge_note_sequences(note_sequences):
    """Merge multiple NoteSequences into one."""
    merged = note_seq.NoteSequence()
    for ns in note_sequences:
        for note in ns.notes:
            new_note = merged.notes.add()
            new_note.CopyFrom(note)
    if note_sequences:
        merged.tempos.extend(note_sequences[0].tempos)
        merged.ticks_per_quarter = note_sequences[0].ticks_per_quarter
    return merged

In [ ]:
#@title Upload Audio
#@markdown Upload a WAV or MP3 of the tango recording.

audio = upload_audio(sample_rate=SAMPLE_RATE)
audio_path = '/tmp/uploaded_audio.wav'
note_seq.audio_io.save_wav_data(audio, sample_rate=SAMPLE_RATE, filename=audio_path)
print(f'Uploaded {len(audio)/SAMPLE_RATE:.1f} seconds of audio.')

In [ ]:
#@title Separate Audio with Demucs
#@markdown Use `htdemucs_6s` to split into drums, bass, other, vocals, guitar, piano.

!python3 -m demucs --name htdemucs_6s --out /tmp/demucs_out {audio_path}

# Locate the separated stems.
stem_dir = Path('/tmp/demucs_out/uploaded_audio/htdemucs_6s')
stems = {path.stem: str(path) for path in stem_dir.glob('*.wav')}
print('Separated stems:', list(stems.keys()))

In [ ]:
#@title Load MT3 Models
#@markdown Load the piano model (`ismir2021`) and the multi-instrument model (`mt3`).

piano_model = InferenceModel('/content/checkpoints/ismir2021/', model_type='ismir2021')
multi_model = InferenceModel('/content/checkpoints/mt3/', model_type='mt3')

print('Models loaded.')

In [ ]:
#@title Transcribe Stems
#@markdown Transcribe each stem with the most appropriate model.

# Map demucs stem names to (model, tango instrument label, program, is_drum).
# The 'other' stem typically contains bandoneón + violin in tango.
STEM_CONFIG = {
    'piano': (piano_model, 'piano', TANGO_PROGRAMS['piano'], False),
    'other': (multi_model, 'bandoneon', TANGO_PROGRAMS['bandoneon'], False),
    'bass': (multi_model, 'bass', TANGO_PROGRAMS['bass'], False),
    'drums': (multi_model, 'drums', TANGO_PROGRAMS['drums'], True),
}

transcribed = {}
for stem_name, (model, label, program, is_drum) in STEM_CONFIG.items():
    if stem_name not in stems:
        print(f'Stem {stem_name!r} not found; skipping.')
        continue
    print(f'Transcribing {stem_name} with {label} program...')
    audio = load_audio(stems[stem_name])
    ns = model(audio)
    ns = assign_program(ns, program, is_drum=is_drum)
    transcribed[stem_name] = ns
    print(f'  -> {len(ns.notes)} notes')

print('Done transcribing stems.')

In [ ]:
#@title Download Combined MIDI
#@markdown Merge all transcribed stems into one MIDI file and download it.

combined_ns = merge_note_sequences(list(transcribed.values()))
note_seq.sequence_proto_to_midi_file(combined_ns, '/tmp/tango_combined.mid')
files.download('/tmp/tango_combined.mid')

print(f'Combined MIDI has {len(combined_ns.notes)} notes.')

In [ ]:
#@title (Optional) Download Per-Stem MIDIs
#@markdown Download a separate MIDI file for each transcribed stem.

for stem_name, ns in transcribed.items():
    out_path = f'/tmp/tango_{stem_name}.mid'
    note_seq.sequence_proto_to_midi_file(ns, out_path)
    files.download(out_path)